In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.io as pio
pio.renderers.default = "browser"
import requests
from scipy.stats import spearmanr

# Add project root to sys.path so we can import our src/index.py module cleanly
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.index import (
    DEFAULT_WEIGHTS,
    BLOCK_INDICATORS,
    compute_composite_scores,
)

DATA_DIR = PROJECT_ROOT / "data"
df = pd.read_csv(DATA_DIR / "clean_em_index.csv")
print(f"Loaded {len(df)} countries across {df['region'].nunique()} regions.")
df[["rank", "country", "region", "income_group_code", "composite_score"]].head(10)

Loaded 104 countries across 6 regions.


,rank,country,region,income_group_code,composite_score
0,1,Malaysia,East Asia & Pacific,UMC,81.47
1,2,Georgia,Europe & Central Asia,UMC,78.04
2,3,China,East Asia & Pacific,UMC,74.41
3,4,Mauritius,Sub-Saharan Africa,UMC,72.83
4,5,Kazakhstan,Europe & Central Asia,UMC,70.99
5,6,Mongolia,East Asia & Pacific,UMC,68.44
6,7,Turkiye,Europe & Central Asia,UMC,68.11
7,8,Armenia,Europe & Central Asia,UMC,67.80
8,9,Thailand,East Asia & Pacific,UMC,67.79
9,10,Viet Nam,East Asia & Pacific,UMC,67.20


In [2]:
block_cols = [
    "score_Growth",
    "score_Demographics",
    "score_Digitalization",
    "score_Business Environment",
    "score_Risk",
]

# 1. Summary statistics by World Bank Income Group
income_summary = (
    df.groupby("income_group")[["composite_score"] + block_cols]
    .mean()
    .round(2)
    .sort_values("composite_score", ascending=False)
)
print("Average Scores by Income Group:")
display(income_summary)

# 2. Spearman correlation matrix across the 5 blocks
# Low-to-moderate correlations confirm each block captures distinct economic dimensions
corr_matrix = df[block_cols].corr(method="spearman").round(2)
fig_corr = px.imshow(
    corr_matrix,
    text_auto=True,
    color_continuous_scale="RdBu_r",
    zmin=-1,
    zmax=1,
    title="Spearman Rank Correlation Across the 5 Index Blocks",
)
fig_corr.show()

Average Scores by Income Group:


,composite_score,score_Growth,score_Demographics,score_Digitalization,score_Business Environment,score_Risk
income_group,,,,,,
Upper middle income,61.60,56.72,61.26,71.18,53.61,69.55
Lower middle income,50.04,46.33,53.03,47.90,42.73,67.05
Low income,40.11,43.06,43.01,23.67,36.31,58.27


In [ ]:
# Pull live forward-looking Real GDP Growth projections (NGDP_RPCH) from IMF DataMapper API
imf_url = "https://www.imf.org/external/datamapper/api/v1/NGDP_RPCH"
try:
    imf_resp = requests.get(imf_url, timeout=20)
    imf_resp.raise_for_status()
    imf_values = imf_resp.json()["values"]["NGDP_RPCH"]

    # Average IMF projected GDP growth over the next 3 forecast years (2026-2028)
    forecast_years = ["2026", "2027", "2028"]
    imf_rows = []
    for iso3, year_dict in imf_values.items():
        vals = [year_dict[y] for y in forecast_years if y in year_dict and year_dict[y] is not None]
        if vals:
            imf_rows.append({"iso3": iso3, "imf_proj_growth_26_28": np.mean(vals)})

    df_imf = pd.DataFrame(imf_rows)
    val_growth = df.merge(df_imf, on="iso3", how="inner")

    rho_g, pval_g = spearmanr(val_growth["score_Growth"], val_growth["imf_proj_growth_26_28"])
    print(f"IMF WEO Validation Sample: {len(val_growth)} matched countries")
    print(f"Spearman Correlation (Growth Score vs. IMF 2026-28 Projected Growth): rho = {rho_g:.3f} (p = {pval_g:.4f})")

    fig_imf = px.scatter(
        val_growth,
        x="imf_proj_growth_26_28",
        y="score_Growth",
        hover_name="country",
        color="income_group",   
        labels={
            "imf_proj_growth_26_28": "IMF WEO Projected Real GDP Growth, 2026-2028 Avg (%)",
            "score_Growth": "Index Growth Block Score (0-100)",
        },
        title=f"External Validation: Growth Score vs. IMF WEO Forecast (Spearman rho = {rho_g:.2f})",
    )
    fig_imf.show()
except Exception as e:
    print(f"IMF API note: {e}")

IMF WEO Validation Sample: 99 matched countries
Spearman Correlation (Growth Score vs. IMF 2026-28 Projected Growth): rho = 0.418 (p = 0.0000)


In [4]:
# Numeric sovereign credit rating scale (S&P / Fitch equivalent: AAA=21, AA+=20, ..., BBB-=12 [Investment Grade cutoff], ..., CCC+=5, D=1)
SOVEREIGN_RATINGS_32 = {
    "CHN": ("A+", 17), "MYS": ("A-", 15), "CHL": ("A", 16), "THA": ("BBB+", 14),
    "PHL": ("BBB+", 14), "PER": ("BBB", 13), "IDN": ("BBB", 13), "KAZ": ("BBB-", 12),
    "MEX": ("BBB-", 12), "IND": ("BBB-", 12), "BGR": ("BBB", 13), "ROU": ("BBB-", 12),
    "COL": ("BB+", 11), "MAR": ("BB+", 11), "VNM": ("BB+", 11), "GEO": ("BB", 10),
    "SRB": ("BB+", 11), "BRA": ("BB", 10), "DOM": ("BB", 10), "ZAF": ("BB-", 9),
    "UZB": ("BB-", 9), "GTM": ("BB", 10), "ARM": ("BB-", 9), "TUR": ("B+", 8),
    "JAM": ("BB-", 9), "EGY": ("B-", 6), "KEN": ("B-", 6), "NGA": ("B-", 6),
    "GHA": ("CCC+", 5), "PAK": ("CCC+", 5), "ARG": ("CCC", 4), "LKA": ("CCC+", 5),
}

ratings_df = pd.DataFrame(
    [{"iso3": k, "rating_label": v[0], "rating_num": v[1]} for k, v in SOVEREIGN_RATINGS_32.items()]
)
val_risk = df.merge(ratings_df, on="iso3", how="inner")

rho_r, pval_r = spearmanr(val_risk["score_Risk"], val_risk["rating_num"])
print(f"Sovereign Credit Rating Validation Sample: {len(val_risk)} major EMs")
print(f"Spearman Correlation (Risk Block Score vs. Numeric Credit Rating): rho = {rho_r:.3f} (p = {pval_r:.4f})")

fig_risk = px.scatter(
    val_risk,
    x="rating_num",
    y="score_Risk",
    text="iso3",
    hover_name="country",
    hover_data=["rating_label", "FP.CPI.TOTL.ZG", "DT.DOD.DECT.GN.ZS", "PV.EST"],
    labels={
        "rating_num": "Numeric Sovereign Credit Rating (1 = D, 12 = BBB- Inv. Grade, 21 = AAA)",
        "score_Risk": "Index Risk Block Score (0-100, Higher = Safer)",
    },
    title=f"External Validation: Risk Score vs. Sovereign Credit Ratings (Spearman rho = {rho_r:.2f})",
)
fig_risk.update_traces(textposition="top center")
fig_risk.show()

# Loose Sanity Check: Where do non-high-income MSCI Emerging Markets countries rank?
msci_em_iso3 = [
    "BRA", "CHN", "COL", "EGY", "IND", "IDN", "MYS", "MEX",
    "PER", "PHL", "ZAF", "THA", "TUR"
]
msci_check = df[df["iso3"].isin(msci_em_iso3)][
    ["rank", "country", "composite_score", "score_Growth", "score_Risk"]
].sort_values("rank")
print("\nMSCI Emerging Markets (non-high-income members) in our 104-country ranking:")
display(msci_check)
print(f"Median rank of major MSCI EM economies: #{msci_check['rank'].median():.0f} out of {len(df)}")

Sovereign Credit Rating Validation Sample: 29 major EMs
Spearman Correlation (Risk Block Score vs. Numeric Credit Rating): rho = 0.533 (p = 0.0029)

MSCI Emerging Markets (non-high-income members) in our 104-country ranking:


,rank,country,composite_score,score_Growth,score_Risk
0,1,Malaysia,81.47,78.69,79.79
2,3,China,74.41,79.80,81.07
6,7,Turkiye,68.11,77.11,56.81
8,9,Thailand,67.79,49.86,78.20
12,13,Colombia,65.52,56.01,65.55
13,14,Peru,65.39,57.79,77.27
14,15,Brazil,65.34,59.61,72.23
17,18,India,63.78,64.63,74.82
18,19,Indonesia,63.74,56.67,75.22
24,25,Mexico,61.55,55.43,71.56


Median rank of major MSCI EM economies: #15 out of 104


In [5]:
rng = np.random.default_rng(seed=42)
n_sims = 500
block_names = list(DEFAULT_WEIGHTS.keys())
base_w = np.array([DEFAULT_WEIGHTS[b] for b in block_names])

# Simulate 500 Dirichlet weight vectors centered on our default weights (concentration = 25)
simulated_weights = rng.dirichlet(alpha=base_w * 25, size=n_sims)

rank_records = {iso3: [] for iso3 in df["iso3"]}
top10_counts = {iso3: 0 for iso3 in df["iso3"]}

for w_vec in simulated_weights:
    w_dict = dict(zip(block_names, w_vec))
    sim_df = compute_composite_scores(df, w_dict)
    for _, row in sim_df.iterrows():
        iso = row["iso3"]
        rk = int(row["rank"])
        rank_records[iso].append(rk)
        if rk <= 10:
            top10_counts[iso] += 1

sens_rows = []
for _, row in df.iterrows():
    iso = row["iso3"]
    ranks = np.array(rank_records[iso])
    sens_rows.append({
        "iso3": iso,
        "country": row["country"],
        "base_rank": int(row["rank"]),
        "composite_score": row["composite_score"],
        "mean_sim_rank": round(float(ranks.mean()), 1),
        "p10_best_rank": int(np.percentile(ranks, 10)),
        "p90_worst_rank": int(np.percentile(ranks, 90)),
        "rank_std": round(float(ranks.std()), 2),
        "top10_prob_pct": round(top10_counts[iso] / n_sims * 100, 1),
    })

sens_df = pd.DataFrame(sens_rows).sort_values("base_rank").reset_index(drop=True)
sens_df.to_csv(DATA_DIR / "sensitivity_summary.csv", index=False)

print("Sensitivity Analysis Saved! Top 15 Countries Stability Across 500 Weight Simulations:")
display(sens_df.head(15))

Sensitivity Analysis Saved! Top 15 Countries Stability Across 500 Weight Simulations:


,iso3,country,base_rank,composite_score,mean_sim_rank,p10_best_rank,p90_worst_rank,rank_std,top10_prob_pct
0,MYS,Malaysia,1,81.47,1.0,1,1,0.00,100.0
1,GEO,Georgia,2,78.04,2.1,2,2,0.23,100.0
2,CHN,China,3,74.41,3.3,3,4,0.60,100.0
3,MUS,Mauritius,4,72.83,4.5,3,6,1.96,98.2
4,KAZ,Kazakhstan,5,70.99,5.6,4,7,1.90,96.2
5,MNG,Mongolia,6,68.44,8.8,6,13,3.75,78.2
6,TUR,Turkiye,7,68.11,9.1,6,14,4.05,73.8
7,ARM,Armenia,8,67.80,9.4,7,12,2.22,80.8
8,THA,Thailand,9,67.79,9.8,5,16,4.32,61.0
9,VNM,Viet Nam,10,67.20,10.2,6,15,3.31,57.2
